In [ ]:
# Cell 1 - Clone private LNPHub repo from dev branch


# Github token

from pathlib import Path
from getpass import getpass
import os
import subprocess

REPO_URL = "https://github.com/Maburidi/lnphub-submission.git"
BRANCH = "dev"
PROJECT_DIR = Path("/content/lnphub-submission")


def get_github_token():
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if token:
        return token
    try:
        from google.colab import userdata
        token = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        token = ""
    return token or getpass("GitHub token for the private repo: ").strip()


def run(cmd, token=None):
    printable = " ".join(map(str, cmd))
    if token:
        printable = printable.replace(token, "***TOKEN***")
    print("$", printable)
    res = subprocess.run(cmd, text=True, capture_output=True)
    out = (res.stdout + res.stderr).replace(token or "", "***TOKEN***")
    if out.strip():
        print(out[-4000:])
    if res.returncode != 0:
        raise RuntimeError(f"Command failed with code {res.returncode}")
    return res


token = get_github_token()
auth_repo_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

if PROJECT_DIR.exists():
    print(f"Repo already exists: {PROJECT_DIR}")
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", auth_repo_url], token=token)
    run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "pull", "origin", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", REPO_URL], token=token)
else:
    run(["git", "clone", "--branch", BRANCH, "--single-branch", auth_repo_url, str(PROJECT_DIR)], token=token)
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", REPO_URL], token=token)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())
run(["git", "-C", str(PROJECT_DIR), "branch", "--show-current"])


## 1. Load Full LNPHub Data


In [ ]:
# Cell 2 - Imports and paths
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

PROJECT_DIR = Path("/content/lnphub-submission")
DATA_DIR = PROJECT_DIR / "data_subsets"
OUT_DIR = PROJECT_DIR / "data_subsets" / "multimodal_expression_3class"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH_CANDIDATES = [
    DATA_DIR / "lnphub_master_v2.0.0.csv",
    PROJECT_DIR / "lnphub_master_v2.0.0.csv",
    Path("/content/lnphub_master_v2.0.0.csv"),
]

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find lnphub_master_v2.0.0.csv in repo or /content.")

print("Using data file:", DATA_PATH)
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Full LNPHub shape:", df.shape)


In [ ]:
# Cell 3 - Check relevant study/metric counts
required = ["study_id", "experimental_setting", "delivery_metric", "delivery", "lnp_id"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

work = df.copy()
work["experimental_setting"] = work["experimental_setting"].astype(str).str.strip()
work["delivery_metric"] = work["delivery_metric"].astype(str).str.strip()
work["study_id"] = work["study_id"].astype(str).str.strip()
work["delivery"] = pd.to_numeric(work["delivery"], errors="coerce")

print("Rows by experimental setting:")
display(work["experimental_setting"].value_counts(dropna=False).reset_index(name="rows").rename(columns={"index": "experimental_setting"}))

print("")
print("Rows by expression-like delivery metric:")
display(
    work[work["delivery_metric"].str.contains("expression", case=False, na=False)]["delivery_metric"]
    .value_counts(dropna=False)
    .reset_index(name="rows")
    .rename(columns={"index": "delivery_metric"})
)


## 2. Select Multimodal Expression Records


In [ ]:
# Cell 4 - Filter expression/expression-discretized records with formulation + physicochemical data
# New multimodal eligibility rule:
#   1) keep expression-like delivery endpoints only
#   2) require a numeric delivery value
#   3) require core formulation molar ratios
#   4) require measured particle size and PDI
#   5) remove studies with 15 or fewer retained datapoints
# No experimental-setting filter and no minimum formulations-per-study filter are applied.
EXPRESSION_METRICS = {"expression", "expression_discretized", "expression_descritized"}
MIN_ROWS_PER_STUDY = 16

FORMULATION_REQUIRED = [
    "il_mol_ratio",
    "helper_lipid_mol_ratio",
    "cholesterol_mol_ratio",
    "peg_lipid_mol_ratio",
]

PHYSICOCHEM_REQUIRED = [
    "particle_size_nm",
    "pdi",
]

MULTIMODAL_REQUIRED = FORMULATION_REQUIRED + PHYSICOCHEM_REQUIRED
missing = [c for c in MULTIMODAL_REQUIRED if c not in work.columns]
if missing:
    raise ValueError(f"Missing required multimodal column(s): {missing}")

candidate = work[
    work["delivery_metric"].isin(EXPRESSION_METRICS)
    & work["delivery"].notna()
].copy()

for col in MULTIMODAL_REQUIRED:
    candidate[col] = pd.to_numeric(candidate[col], errors="coerce")

candidate["has_core_formulation"] = candidate[FORMULATION_REQUIRED].notna().all(axis=1)
candidate["has_physicochemical"] = candidate[PHYSICOCHEM_REQUIRED].notna().all(axis=1)
candidate["has_complete_multimodal"] = candidate[MULTIMODAL_REQUIRED].notna().all(axis=1)

subset_before_study_filter = candidate[candidate["has_complete_multimodal"]].copy()
study_n = subset_before_study_filter["study_id"].value_counts()
keep_studies = study_n[study_n >= MIN_ROWS_PER_STUDY].index
excluded_small_studies = study_n[study_n < MIN_ROWS_PER_STUDY].rename_axis("study_id").reset_index(name="retained_rows")
subset = subset_before_study_filter[subset_before_study_filter["study_id"].isin(keep_studies)].copy()

study_summary = (
    candidate.groupby(["experimental_setting", "study_id"], dropna=False)
    .agg(
        expression_rows=("study_id", "size"),
        complete_multimodal_rows=("has_complete_multimodal", "sum"),
        rows_with_core_formulation=("has_core_formulation", "sum"),
        rows_with_physicochemical=("has_physicochemical", "sum"),
        formulations=("lnp_id", "nunique"),
        complete_multimodal_formulations=(
            "lnp_id",
            lambda s: s[candidate.loc[s.index, "has_complete_multimodal"]].nunique(),
        ),
        metrics=("delivery_metric", lambda s: ", ".join(sorted(s.dropna().unique()))),
        delivery_min=("delivery", "min"),
        delivery_median=("delivery", "median"),
        delivery_max=("delivery", "max"),
        delivery_unique_values=("delivery", "nunique"),
    )
    .reset_index()
    .sort_values("complete_multimodal_rows", ascending=False)
)
study_summary["retained_after_min_rows_filter"] = study_summary["study_id"].isin(keep_studies)

print(f"Expression-like rows with numeric delivery: {len(candidate):,}")
print(f"Rows with complete formulation + particle size/PDI: {len(subset_before_study_filter):,}")
print(f"Rows retained after removing studies with <=15 datapoints: {len(subset):,}")
print(f"Studies retained: {subset['study_id'].nunique():,}")
print(f"Unique formulations retained: {subset['lnp_id'].nunique():,}")
print("")
print("Rows retained by experimental setting:")
display(subset["experimental_setting"].value_counts(dropna=False).reset_index(name="rows"))
print("")
print("Rows retained by delivery metric:")
display(subset["delivery_metric"].value_counts(dropna=False).reset_index(name="rows"))
print("")
print("Excluded studies with <=15 complete multimodal rows:")
display(excluded_small_studies)
print("")
print("Multimodal coverage by study:")
display(study_summary[study_summary["complete_multimodal_rows"] > 0])


In [ ]:
# Plot continuous expression delivery distributions before 3-class discretization
# Excludes already discretized rows and LNPhub_FG_2026
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

d = subset.copy()
d["delivery_metric"] = d["delivery_metric"].astype(str).str.strip()
d["study_id"] = d["study_id"].astype(str).str.strip()
d["delivery"] = pd.to_numeric(d["delivery"], errors="coerce")

d = d[
    (d["delivery_metric"] == "expression")
    & (d["study_id"] != "LNPhub_FG_2026")
    & d["study_id"].notna()
    & d["delivery"].notna()
].copy()

summary = (
    d.groupby("study_id")["delivery"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        q05=lambda x: x.quantile(0.05),
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        q95=lambda x: x.quantile(0.95),
        max="max",
        frac_z_gt_1=lambda x: (x > 1).mean(),
        frac_z_gt_2=lambda x: (x > 2).mean(),
        frac_z_lt_minus1=lambda x: (x < -1).mean(),
        frac_abs_lt_025=lambda x: (x.abs() < 0.25).mean(),
    )
    .reset_index()
)

summary["mean_ok"] = summary["mean"].abs() < 0.10
summary["std_ok"] = summary["std"].between(0.90, 1.10)
summary["normalization_flag"] = np.select(
    [
        summary["n"] < 20,
        summary["std"].fillna(0) < 0.25,
        ~summary["mean_ok"] & ~summary["std_ok"],
        ~summary["mean_ok"],
        ~summary["std_ok"],
    ],
    [
        "small_n",
        "very_low_variance",
        "mean_and_std_off",
        "mean_off",
        "std_off",
    ],
    default="ok",
)

print("Continuous expression rows:", len(d))
print("Continuous expression studies:", d["study_id"].nunique())

print("\nOverall continuous delivery summary:")
display(d["delivery"].agg(["count", "mean", "median", "std", "min", "max"]).to_frame().T)

print("\nPer-study continuous normalization QC:")
display(summary.sort_values(["normalization_flag", "n"], ascending=[True, False]))

print("\nPotential problem continuous studies:")
display(summary[summary["normalization_flag"] != "ok"].sort_values(["normalization_flag", "n"], ascending=[True, False]))

order = (
    d.groupby("study_id")["lnp_id"]
    .nunique()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(9, max(4, 0.30 * d["study_id"].nunique())))
sns.boxplot(
    data=d,
    y="study_id",
    x="delivery",
    order=order,
    color="#4E79A7",
    fliersize=1.5,
    linewidth=0.8,
)
plt.axvline(0, color="black", lw=0.8)
plt.axvline(1, color="#D55E00", lw=0.8, ls="--")
plt.axvline(2, color="#D55E00", lw=0.8, ls=":")
plt.xlabel("Normalized continuous delivery before discretization")
plt.ylabel("")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.2, 3.8))
sns.histplot(
    data=d,
    x="delivery",
    bins=80,
    element="step",
    stat="density",
    color="#4E79A7",
)
plt.axvline(0, color="black", lw=0.8)
plt.axvline(1, color="#D55E00", lw=0.8, ls="--")
plt.axvline(2, color="#D55E00", lw=0.8, ls=":")
plt.xlabel("Normalized continuous delivery before discretization")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

## 3. Harmonize Delivery Into Three Classes


In [ ]:
# Cell 5 - Create harmonized 3-class delivery label: 0=low, 1=medium, 2=high
# Continuous expression uses fixed thresholds on normalized delivery values:
#   low    = delivery < 0
#   medium = 0 <= delivery < 1
#   high   = delivery >= 1
# This does not force equal class sizes; high-delivery LNPs remain rare if they are rare in the data.
CLASS_LABELS = {0: "low", 1: "medium", 2: "high"}
LOW_HIGH_CUTOFF = 0.0
MEDIUM_HIGH_CUTOFF = 1.0


def value_threshold_classes(values):
    values = pd.to_numeric(values, errors="coerce")
    out = pd.Series(pd.NA, index=values.index, dtype="Int64")
    out[values < LOW_HIGH_CUTOFF] = 0
    out[(values >= LOW_HIGH_CUTOFF) & (values < MEDIUM_HIGH_CUTOFF)] = 1
    out[values >= MEDIUM_HIGH_CUTOFF] = 2
    return out


def collapse_discrete_to_three(values):
    values = pd.to_numeric(values, errors="coerce")
    counts = values.value_counts(dropna=True)
    groups = [[float(v)] for v in sorted(counts.index)]

    while len(groups) > 3:
        centers = [np.mean(g) for g in groups]
        distances = [centers[i + 1] - centers[i] for i in range(len(groups) - 1)]
        min_dist = min(distances)
        tied = [i for i, d in enumerate(distances) if np.isclose(d, min_dist)]

        def merged_count(i):
            return sum(int(counts.loc[v]) for v in groups[i] + groups[i + 1])

        merge_i = min(tied, key=merged_count)
        groups[merge_i] = groups[merge_i] + groups[merge_i + 1]
        del groups[merge_i + 1]

    mapping = {}
    for cls, group in enumerate(groups):
        for v in group:
            mapping[v] = cls
    return values.astype(float).map(mapping).astype("Int64"), mapping


class_parts = []
mapping_rows = []

for (study_id, metric), sdf in subset.groupby(["study_id", "delivery_metric"], sort=False):
    sdf = sdf.copy()
    n_unique = sdf["delivery"].nunique(dropna=True)

    if metric in {"expression_discretized", "expression_descritized"}:
        if n_unique == 3:
            ordered = sorted(pd.to_numeric(sdf["delivery"], errors="coerce").dropna().unique())
            mapping = {float(v): i for i, v in enumerate(ordered)}
            sdf["delivery_class_3"] = pd.to_numeric(sdf["delivery"], errors="coerce").astype(float).map(mapping).astype("Int64")
            rule = "already_3_discrete_levels_preserved_order"
        elif n_unique in {4, 5}:
            sdf["delivery_class_3"], mapping = collapse_discrete_to_three(sdf["delivery"])
            rule = f"collapsed_{n_unique}_discrete_levels_to_3_by_closest_adjacent_values"
        else:
            sdf["delivery_class_3"] = value_threshold_classes(sdf["delivery"])
            mapping = {}
            rule = f"unexpected_{n_unique}_discrete_levels_used_fixed_delivery_thresholds"
    else:
        sdf["delivery_class_3"] = value_threshold_classes(sdf["delivery"])
        mapping = {
            "low": f"delivery < {LOW_HIGH_CUTOFF}",
            "medium": f"{LOW_HIGH_CUTOFF} <= delivery < {MEDIUM_HIGH_CUTOFF}",
            "high": f"delivery >= {MEDIUM_HIGH_CUTOFF}",
        }
        rule = "continuous_expression_fixed_delivery_thresholds"

    sdf["delivery_class_label"] = sdf["delivery_class_3"].map(CLASS_LABELS)
    class_parts.append(sdf)
    mapping_rows.append({
        "study_id": study_id,
        "delivery_metric": metric,
        "n_rows": len(sdf),
        "n_formulations": sdf["lnp_id"].nunique(),
        "n_unique_delivery_values": int(n_unique),
        "delivery_mean": sdf["delivery"].mean(),
        "delivery_std": sdf["delivery"].std(),
        "delivery_min": sdf["delivery"].min(),
        "delivery_max": sdf["delivery"].max(),
        "harmonization_rule": rule,
        "discrete_value_to_class_mapping": mapping,
    })

subset_3class = pd.concat(class_parts, ignore_index=True)
mapping_table = pd.DataFrame(mapping_rows)

print("Harmonization rules by study/metric:")
display(mapping_table)

print("")
print("Class sanity check:")
display(
    subset_3class.groupby(["study_id", "delivery_metric", "delivery_class_3", "delivery_class_label"])
    .size()
    .reset_index(name="rows")
    .sort_values(["study_id", "delivery_metric", "delivery_class_3"])
)


## 4. Save Subset and Source Tables


In [ ]:
# Cell 6 - Save harmonized subset and summary tables
subset_path = OUT_DIR / "lnphub_expression_multimodal_physchem_formulation_3class.csv"
study_summary_path = OUT_DIR / "multimodal_eligibility_summary.csv"
mapping_path = OUT_DIR / "delivery_3class_harmonization_rules.csv"
class_counts_path = OUT_DIR / "study_class_counts.csv"

class_counts = (
    subset_3class.groupby(["study_id", "delivery_class_3", "delivery_class_label"])
    .agg(rows=("study_id", "size"), formulations=("lnp_id", "nunique"))
    .reset_index()
    .sort_values(["study_id", "delivery_class_3"])
)

subset_3class.to_csv(subset_path, index=False)
study_summary.to_csv(study_summary_path, index=False)
mapping_table.to_csv(mapping_path, index=False)
class_counts.to_csv(class_counts_path, index=False)

print("Saved subset:", subset_path)
print("Saved study summary:", study_summary_path)
print("Saved harmonization rules:", mapping_path)
print("Saved class counts:", class_counts_path)
print("")
print("Final subset shape:", subset_3class.shape)
print("Studies:", subset_3class["study_id"].nunique())
print("Unique formulations:", subset_3class["lnp_id"].nunique())

display(class_counts)


## 5. Plot Per-Study Class Balance


In [ ]:
# Cell 7 - Bar plots of low / medium / high class counts per study
plot_df = class_counts.copy()
plot_df["class"] = pd.Categorical(plot_df["delivery_class_label"], categories=["low", "medium", "high"], ordered=True)

order = (
    subset_3class.groupby("study_id")["lnp_id"]
    .nunique()
    .sort_values(ascending=False)
    .index
    .tolist()
)

palette = {"low": "#4E79A7", "medium": "#A0A7B4", "high": "#D55E00"}

fig, ax = plt.subplots(figsize=(9.2, max(4.2, 0.34 * len(order))))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

sns.barplot(
    data=plot_df,
    y="study_id",
    x="rows",
    hue="class",
    order=order,
    hue_order=["low", "medium", "high"],
    palette=palette,
    ax=ax,
    edgecolor="white",
    linewidth=0.5,
)

ax.set_xlabel("Rows")
ax.set_ylabel("")
ax.grid(axis="x", color="#E5E7EB", linewidth=0.6)
ax.set_axisbelow(True)
ax.legend(title="3-class delivery", frameon=False, loc="lower right")
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout()
fig_path = FIG_DIR / "multimodal_expression_3class_counts_by_study.pdf"
fig.savefig(fig_path, bbox_inches="tight")
print("Saved figure:", fig_path)
plt.show()


In [ ]:
# Cell 8 - Faceted per-study distribution view
plot_df = subset_3class.copy()
plot_df["delivery_class_label"] = pd.Categorical(
    plot_df["delivery_class_label"], categories=["low", "medium", "high"], ordered=True
)

n_studies = plot_df["study_id"].nunique()
ncols = 3
nrows = int(np.ceil(n_studies / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(9.2, max(3.2, 2.05 * nrows)), squeeze=False)
fig.patch.set_facecolor("white")

for ax in axes.flat:
    ax.axis("off")

for ax, study_id in zip(axes.flat, order):
    sdf = plot_df[plot_df["study_id"] == study_id]
    counts = sdf["delivery_class_label"].value_counts().reindex(["low", "medium", "high"], fill_value=0)
    ax.axis("on")
    ax.bar(counts.index.astype(str), counts.values, color=[palette[k] for k in counts.index], width=0.72)
    ax.set_title(study_id, fontsize=9, loc="left", pad=3)
    ax.tick_params(axis="x", labelsize=8)
    ax.tick_params(axis="y", labelsize=7)
    ax.grid(axis="y", color="#E5E7EB", linewidth=0.5)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for x, y in enumerate(counts.values):
        ax.text(x, y, f"{int(y):,}", ha="center", va="bottom", fontsize=7)

fig.tight_layout(h_pad=1.0, w_pad=1.0)
fig_path = FIG_DIR / "multimodal_expression_3class_counts_faceted.pdf"
fig.savefig(fig_path, bbox_inches="tight")
print("Saved figure:", fig_path)
plt.show()


# 5. Balance Classes Within Each Study
This creates a training-ready balanced subset by downsampling majority classes separately within each study. The full 3-class subset remains unchanged.


In [ ]:
# Cell 7 - Balance low / medium / high class counts within each study
# Strategy: for each study, randomly downsample each class to the size of that study's smallest class.
# This keeps class balance study-aware and avoids letting large low-delivery classes dominate training.

BALANCE_RANDOM_SEED = 7
BALANCE_TARGET_MULTIPLIER = 1.0  # 1.0 = exactly the smallest class; change to 1.05 or 1.10 if you want "close" but slightly larger.

rng = np.random.default_rng(BALANCE_RANDOM_SEED)
balanced_parts = []
balance_rows = []

for study_id, sdf in subset_3class.groupby("study_id", sort=False):
    sdf = sdf.dropna(subset=["delivery_class_3"]).copy()
    counts = sdf["delivery_class_3"].value_counts().reindex([0, 1, 2], fill_value=0).astype(int)

    positive_counts = counts[counts > 0]
    if len(positive_counts) < 3:
        balance_rows.append({
            "study_id": study_id,
            "status": "skipped_missing_class",
            "original_low": int(counts.loc[0]),
            "original_medium": int(counts.loc[1]),
            "original_high": int(counts.loc[2]),
            "target_per_class": np.nan,
            "balanced_rows": 0,
            "discarded_rows": int(len(sdf)),
        })
        continue

    min_class_n = int(positive_counts.min())
    target_n = int(np.ceil(min_class_n * BALANCE_TARGET_MULTIPLIER))

    sampled = []
    sampled_counts = {}
    for cls in [0, 1, 2]:
        cls_df = sdf[sdf["delivery_class_3"] == cls]
        take_n = min(len(cls_df), target_n)
        chosen_idx = rng.choice(cls_df.index.to_numpy(), size=take_n, replace=False)
        sampled.append(cls_df.loc[chosen_idx])
        sampled_counts[cls] = take_n

    balanced_study = pd.concat(sampled, ignore_index=False).sample(frac=1, random_state=BALANCE_RANDOM_SEED).reset_index(drop=True)
    balanced_parts.append(balanced_study)

    balance_rows.append({
        "study_id": study_id,
        "status": "balanced",
        "original_low": int(counts.loc[0]),
        "original_medium": int(counts.loc[1]),
        "original_high": int(counts.loc[2]),
        "target_per_class": int(target_n),
        "balanced_low": int(sampled_counts[0]),
        "balanced_medium": int(sampled_counts[1]),
        "balanced_high": int(sampled_counts[2]),
        "original_rows": int(len(sdf)),
        "balanced_rows": int(len(balanced_study)),
        "discarded_rows": int(len(sdf) - len(balanced_study)),
    })

balanced_subset_3class = pd.concat(balanced_parts, ignore_index=True) if balanced_parts else pd.DataFrame(columns=subset_3class.columns)
balanced_subset_3class = balanced_subset_3class.sample(frac=1, random_state=BALANCE_RANDOM_SEED).reset_index(drop=True)

MIN_BALANCED_ROWS_PER_CLASS = 15

balanced_class_n = (
    balanced_subset_3class
    .groupby(["study_id", "delivery_class_3"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[0, 1, 2], fill_value=0)
)

keep_balanced_studies = balanced_class_n[
    balanced_class_n.min(axis=1) >= MIN_BALANCED_ROWS_PER_CLASS
].index
excluded_low_balanced_class_studies = (
    balanced_class_n.loc[~balanced_class_n.index.isin(keep_balanced_studies)]
    .rename(columns={0: "low", 1: "medium", 2: "high"})
    .reset_index()
)

balanced_subset_3class = balanced_subset_3class[
    balanced_subset_3class["study_id"].isin(keep_balanced_studies)
].copy().reset_index(drop=True)

balance_summary = pd.DataFrame(balance_rows)
balance_summary["retained_after_min_15_per_class_filter"] = balance_summary["study_id"].isin(keep_balanced_studies)

balanced_class_counts = (
    balanced_subset_3class.groupby(["study_id", "delivery_class_3", "delivery_class_label"])
    .agg(rows=("study_id", "size"), formulations=("lnp_id", "nunique"))
    .reset_index()
    .sort_values(["study_id", "delivery_class_3"])
)

balanced_subset_path = OUT_DIR / "lnphub_expression_multimodal_physchem_formulation_3class_balanced_by_study.csv"
balance_summary_path = OUT_DIR / "study_class_balancing_summary.csv"
balanced_class_counts_path = OUT_DIR / "study_class_counts_balanced.csv"

balanced_subset_3class.to_csv(balanced_subset_path, index=False)
balance_summary.to_csv(balance_summary_path, index=False)
balanced_class_counts.to_csv(balanced_class_counts_path, index=False)

print("Saved balanced subset:", balanced_subset_path)
print("Saved balancing summary:", balance_summary_path)
print("Saved balanced class counts:", balanced_class_counts_path)
print("")
print("Full 3-class subset rows:", f"{len(subset_3class):,}")
print("Balanced subset rows:", f"{len(balanced_subset_3class):,}")
print("Rows discarded during balancing:", f"{len(subset_3class) - len(balanced_subset_3class):,}")
print("Balanced studies:", balanced_subset_3class["study_id"].nunique())
print(f"Minimum required rows per class after balancing: {MIN_BALANCED_ROWS_PER_CLASS}")

print("")
print("Studies removed after balancing because at least one class had <15 rows:")
display(excluded_low_balanced_class_studies)

print("")
print("Balancing summary:")
display(balance_summary.sort_values("balanced_rows", ascending=False))

print("")
print("Balanced class counts:")
display(balanced_class_counts)


In [ ]:
# Cell 8 - Plot balanced low / medium / high class counts per study
plot_df = balanced_class_counts.copy()
plot_df["class"] = pd.Categorical(plot_df["delivery_class_label"], categories=["low", "medium", "high"], ordered=True)

balanced_order = (
    balanced_subset_3class.groupby("study_id")["lnp_id"]
    .nunique()
    .sort_values(ascending=False)
    .index
    .tolist()
)

palette = {"low": "#4E79A7", "medium": "#A0A7B4", "high": "#D55E00"}

fig, ax = plt.subplots(figsize=(9.2, max(4.2, 0.34 * len(balanced_order))))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

sns.barplot(
    data=plot_df,
    y="study_id",
    x="rows",
    hue="class",
    order=balanced_order,
    hue_order=["low", "medium", "high"],
    palette=palette,
    ax=ax,
    edgecolor="white",
    linewidth=0.5,
)

ax.set_xlabel("Rows after within-study downsampling")
ax.set_ylabel("")
ax.grid(axis="x", color="#E5E7EB", linewidth=0.6)
ax.set_axisbelow(True)
ax.legend(title="3-class delivery", frameon=False, loc="lower right")
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout()
fig_path = FIG_DIR / "multimodal_expression_3class_counts_by_study_balanced.pdf"
fig.savefig(fig_path, bbox_inches="tight")
print("Saved figure:", fig_path)
plt.show()


In [ ]:
print("Balanced samples:", len(balanced_subset_3class))
print("Balanced studies:", balanced_subset_3class["study_id"].nunique())
print("Balanced formulations:", balanced_subset_3class["lnp_id"].nunique())

#6. Hold-Out Split and Molecular Fingerprint / Descriptor Classifiers
This section uses the balanced 3-class subset. LNPhub_YZ_2024 and LNPhub_BL_2024 are kept as a completely held-out study-level test set, while the remaining studies receive 5 study-aware CV folds.

In [ ]:
# Cell 9 - Create held-out test set and leave-one-study-out CV split for multimodal 3-class classification
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

CLASS_RANDOM_SEED = 7
HOLDOUT_STUDIES = ["LNPhub_LZ_2025", "LNPhub_GN_2025"]

CLASS_SPLIT_DIR = OUT_DIR / "classification_splits"
CLASS_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

classifier_data = balanced_subset_3class.copy()
classifier_data["study_id"] = classifier_data["study_id"].astype(str).str.strip()
classifier_data["delivery_class_3"] = pd.to_numeric(classifier_data["delivery_class_3"], errors="coerce")
classifier_data = classifier_data[classifier_data["delivery_class_3"].isin([0, 1, 2])].copy()
classifier_data["delivery_class_3"] = classifier_data["delivery_class_3"].astype(int)

missing_holdouts = sorted(set(HOLDOUT_STUDIES) - set(classifier_data["study_id"].unique()))
if missing_holdouts:
    raise ValueError(f"Hold-out studies not found in balanced multimodal data: {missing_holdouts}. Choose holdout studies from the displayed balanced-study list above.")

test_mask = classifier_data["study_id"].isin(HOLDOUT_STUDIES)
test_classification = classifier_data[test_mask].copy().reset_index(drop=True)
trainval_classification = classifier_data[~test_mask].copy().reset_index(drop=True)

# Leave-one-study-out CV: each remaining study is one validation fold.
trainval_studies = sorted(trainval_classification["study_id"].dropna().unique())
study_to_fold = {study_id: fold for fold, study_id in enumerate(trainval_studies)}
trainval_classification["cv_fold"] = trainval_classification["study_id"].map(study_to_fold).astype(int)

test_classification["cv_fold"] = -1
trainval_classification["oracle_split"] = "trainval"
test_classification["oracle_split"] = "untouched_test"

TRAINVAL_CLASS_PATH = CLASS_SPLIT_DIR / "lnphub_expression_multimodal_3class_balanced_trainval_leave_one_study_out.csv"
TEST_CLASS_PATH = CLASS_SPLIT_DIR / "lnphub_expression_multimodal_3class_balanced_untouched_test.csv"
SPLIT_SUMMARY_PATH = CLASS_SPLIT_DIR / "classification_split_summary_leave_one_study_out.csv"

trainval_classification.to_csv(TRAINVAL_CLASS_PATH, index=False)
test_classification.to_csv(TEST_CLASS_PATH, index=False)

split_summary = pd.concat([trainval_classification, test_classification], ignore_index=True)
split_summary_table = (
    split_summary.groupby(["oracle_split", "study_id", "delivery_class_3", "delivery_class_label"])
    .size()
    .reset_index(name="rows")
    .sort_values(["oracle_split", "study_id", "delivery_class_3"])
)
split_summary_table.to_csv(SPLIT_SUMMARY_PATH, index=False)

print("Saved trainval split:", TRAINVAL_CLASS_PATH)
print("Saved untouched test split:", TEST_CLASS_PATH)
print("Saved split summary:", SPLIT_SUMMARY_PATH)
print("")
print("Trainval rows:", f"{len(trainval_classification):,}")
print("Untouched test rows:", f"{len(test_classification):,}")
print("Trainval studies:", trainval_classification["study_id"].nunique())
print("Untouched test studies:", test_classification["study_id"].nunique(), HOLDOUT_STUDIES)
print("CV strategy: leave-one-study-out")
print("CV folds:", trainval_classification["cv_fold"].nunique())

print("")
print("Rows by split/class:")
display(
    split_summary.groupby(["oracle_split", "delivery_class_3", "delivery_class_label"])
    .size()
    .reset_index(name="rows")
)

print("")
print("Leave-one-study-out fold summary:")
display(
    trainval_classification.groupby("cv_fold")
    .agg(
        validation_study=("study_id", lambda s: s.iloc[0]),
        rows=("study_id", "size"),
        studies=("study_id", "nunique"),
    )
    .reset_index()
)

print("")
print("CV fold x class counts:")
display(
    trainval_classification.groupby(["cv_fold", "study_id", "delivery_class_3"])
    .size()
    .reset_index(name="rows")
    .pivot_table(index=["cv_fold", "study_id"], columns="delivery_class_3", values="rows", fill_value=0)
    .reset_index()
)

print("")
print("Held-out test studies/classes:")
display(
    test_classification.groupby(["study_id", "delivery_class_3", "delivery_class_label"])
    .size()
    .reset_index(name="rows")
    .sort_values(["study_id", "delivery_class_3"])
)


## 7. Molecular Fingerprint / Descriptor + Formulation / Physicochemical Classifiers


In [ ]:
# Cell 10 - Train multimodal molecular fingerprint / descriptor 3-class classifiers
# Feature blocks:
#   molecular: ECFP4 or RDKit descriptors from ionizable-lipid SMILES
#   multimodal: formulation molar ratios + measured particle size/PDI
# Models:
#   MLP, XGBoost, Random Forest for each molecular feature block
import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display

MULTIMODAL_TABULAR_OUT = (
    PROJECT_DIR
    / "results"
    / "multimodal_expression_3class_classification_balanced"
    / "molecular_fingerprint_descriptor"
)
MULTIMODAL_TABULAR_OUT.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_DIR / "src"
TRAINER_PATH = SRC_DIR / "lnphub_oracles" / "train_tabular_multimodal_classification.py"
if not TRAINER_PATH.exists():
    print("Multimodal trainer not found; pulling latest dev branch...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", "dev"], check=False)

if not TRAINER_PATH.exists():
    raise FileNotFoundError(
        f"Missing {TRAINER_PATH}. Pull the latest dev branch or rerun the clone/setup cell before training."
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


def ensure_import(import_name, pip_names):
    try:
        __import__(import_name)
        return
    except ModuleNotFoundError:
        pass

    if isinstance(pip_names, str):
        pip_names = [pip_names]

    last_error = None
    for pip_name in pip_names:
        print(f"Installing missing package for {import_name}: {pip_name}")
        res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], text=True, capture_output=True)
        if res.returncode != 0:
            last_error = (res.stdout + res.stderr)[-4000:]
            print(last_error)
            continue
        try:
            __import__(import_name)
            return
        except ModuleNotFoundError as exc:
            last_error = repr(exc)

    raise ModuleNotFoundError(f"Could not import {import_name} after installing {pip_names}. Last error: {last_error}")


# Required by ECFP4/RDKit descriptors and by parquet output. XGBoost is used by two models.
ensure_import("rdkit", ["rdkit", "rdkit-pypi"])
ensure_import("pyarrow", "pyarrow")
ensure_import("sklearn", "scikit-learn")
ensure_import("xgboost", "xgboost")

print("Using trainer:", TRAINER_PATH)

MULTIMODAL_FEATURE_COLS = [
    "il_mol_ratio",
    "helper_lipid_mol_ratio",
    "cholesterol_mol_ratio",
    "peg_lipid_mol_ratio",
    "particle_size_nm",
    "pdi",
]

TABULAR_MULTIMODAL_METHODS = [
    "ecfp4_multimodal_mlp",
    "ecfp4_multimodal_xgboost",
    "ecfp4_multimodal_random_forest",
    "rdkit_descriptors_multimodal_mlp",
    "rdkit_descriptors_multimodal_xgboost",
    "rdkit_descriptors_multimodal_random_forest",
]

required_paths = [TRAINVAL_CLASS_PATH, TEST_CLASS_PATH]
missing_paths = [str(p) for p in required_paths if not Path(p).exists()]
if missing_paths:
    raise FileNotFoundError(
        "Run Cell 9 first to create the balanced multimodal train/test split. Missing: "
        + ", ".join(missing_paths)
    )

trainval_preview = pd.read_csv(TRAINVAL_CLASS_PATH, low_memory=False)
missing_cols = [c for c in MULTIMODAL_FEATURE_COLS if c not in trainval_preview.columns]
if missing_cols:
    raise ValueError(f"Missing multimodal columns in trainval split: {missing_cols}")

print("Trainval split:", TRAINVAL_CLASS_PATH)
print("Untouched test split:", TEST_CLASS_PATH)
print("Output folder:", MULTIMODAL_TABULAR_OUT)
print("Multimodal covariates:", ", ".join(MULTIMODAL_FEATURE_COLS))
print("Trainval rows:", f"{len(trainval_preview):,}")
print("Trainval studies:", trainval_preview["study_id"].nunique())
print("Trainval class counts:")
display(trainval_preview["delivery_class_3"].value_counts().sort_index().reset_index(name="rows"))

cmd = [
    sys.executable,
    "-u",
    "-m",
    "lnphub_oracles.train_tabular_multimodal_classification",
    "--trainval-subset", str(TRAINVAL_CLASS_PATH),
    "--test-subset", str(TEST_CLASS_PATH),
    "--output-dir", str(MULTIMODAL_TABULAR_OUT),
    "--target-col", "delivery_class_3",
    "--multimodal-cols", *MULTIMODAL_FEATURE_COLS,
    "--methods", *TABULAR_MULTIMODAL_METHODS,
]

print("\nRunning:", " ".join(cmd))
print("\n--- live multimodal tabular training output ---\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + env.get("PYTHONPATH", "")
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Multimodal tabular classifier training failed with exit code {return_code}")

manifest_path = MULTIMODAL_TABULAR_OUT / "tabular_multimodal_classification_manifest.csv"
metrics_path = MULTIMODAL_TABULAR_OUT / "tabular_multimodal_all_metrics.csv"

print("\nSaved manifest:", manifest_path)
print("Saved all metrics:", metrics_path)

manifest = pd.read_csv(manifest_path)
display(manifest)

print("\nUntouched-test summary:")
display(
    manifest[[
        "method",
        "test_accuracy",
        "test_balanced_accuracy",
        "test_macro_f1",
        "test_roc_auc_ovr_macro",
        "test_average_precision_macro",
    ]].sort_values("test_balanced_accuracy", ascending=False)
)


## 8. Random-Split Molecular Fingerprint / Descriptor + Formulation / Physicochemical Classifiers


In [ ]:
# Cell 11 - Random train/validation/test split multimodal tabular classifiers
# This does NOT use study-held-out CV. It shuffles the balanced 3-class dataset,
# makes a stratified random train/validation/test split, then trains the same six models once.
import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

RANDOM_SPLIT_SEED = 7
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

RANDOM_SPLIT_DIR = OUT_DIR / "classification_random_splits"
RANDOM_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_TRAIN_PATH = RANDOM_SPLIT_DIR / "lnphub_expression_multimodal_3class_balanced_random_train.csv"
RANDOM_VALIDATION_PATH = RANDOM_SPLIT_DIR / "lnphub_expression_multimodal_3class_balanced_random_validation.csv"
RANDOM_TEST_PATH = RANDOM_SPLIT_DIR / "lnphub_expression_multimodal_3class_balanced_random_test.csv"
RANDOM_SPLIT_SUMMARY_PATH = RANDOM_SPLIT_DIR / "random_split_summary.csv"

RANDOM_MULTIMODAL_TABULAR_OUT = (
    PROJECT_DIR
    / "results"
    / "multimodal_expression_3class_classification_balanced"
    / "molecular_fingerprint_descriptor_random_split"
)
RANDOM_MULTIMODAL_TABULAR_OUT.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_DIR / "src"
TRAINER_PATH = SRC_DIR / "lnphub_oracles" / "train_tabular_multimodal_random_split_classification.py"
if not TRAINER_PATH.exists():
    print("Random-split multimodal trainer not found; pulling latest dev branch...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", "dev"], check=False)

if not TRAINER_PATH.exists():
    raise FileNotFoundError(
        f"Missing {TRAINER_PATH}. Pull the latest dev branch or rerun the clone/setup cell before training."
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


def ensure_import(import_name, pip_names):
    try:
        __import__(import_name)
        return
    except ModuleNotFoundError:
        pass

    if isinstance(pip_names, str):
        pip_names = [pip_names]

    last_error = None
    for pip_name in pip_names:
        print(f"Installing missing package for {import_name}: {pip_name}")
        res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], text=True, capture_output=True)
        if res.returncode != 0:
            last_error = (res.stdout + res.stderr)[-4000:]
            print(last_error)
            continue
        try:
            __import__(import_name)
            return
        except ModuleNotFoundError as exc:
            last_error = repr(exc)

    raise ModuleNotFoundError(f"Could not import {import_name} after installing {pip_names}. Last error: {last_error}")


ensure_import("rdkit", ["rdkit", "rdkit-pypi"])
ensure_import("pyarrow", "pyarrow")
ensure_import("sklearn", "scikit-learn")
ensure_import("xgboost", "xgboost")

MULTIMODAL_FEATURE_COLS = [
    "il_mol_ratio",
    "helper_lipid_mol_ratio",
    "cholesterol_mol_ratio",
    "peg_lipid_mol_ratio",
    "particle_size_nm",
    "pdi",
]

TABULAR_MULTIMODAL_METHODS = [
    "ecfp4_multimodal_mlp",
    "ecfp4_multimodal_xgboost",
    "ecfp4_multimodal_random_forest",
    "rdkit_descriptors_multimodal_mlp",
    "rdkit_descriptors_multimodal_xgboost",
    "rdkit_descriptors_multimodal_random_forest",
]

if "balanced_subset_3class" in globals():
    random_source = balanced_subset_3class.copy()
elif "balanced_subset_path" in globals() and Path(balanced_subset_path).exists():
    random_source = pd.read_csv(balanced_subset_path, low_memory=False)
else:
    candidate_path = OUT_DIR / "lnphub_expression_multimodal_physchem_formulation_3class_balanced_by_study.csv"
    if not candidate_path.exists():
        raise FileNotFoundError("Run the balancing cell first, or provide the saved balanced subset file.")
    random_source = pd.read_csv(candidate_path, low_memory=False)

random_source = random_source.copy()
random_source["delivery_class_3"] = pd.to_numeric(random_source["delivery_class_3"], errors="coerce")
random_source = random_source[random_source["delivery_class_3"].isin([0, 1, 2])].copy()
random_source["delivery_class_3"] = random_source["delivery_class_3"].astype(int)
random_source = random_source.sample(frac=1, random_state=RANDOM_SPLIT_SEED).reset_index(drop=True)

missing_cols = [c for c in MULTIMODAL_FEATURE_COLS if c not in random_source.columns]
if missing_cols:
    raise ValueError(f"Missing multimodal columns in balanced source data: {missing_cols}")

train_df, temp_df = train_test_split(
    random_source,
    test_size=(VALIDATION_FRACTION + TEST_FRACTION),
    random_state=RANDOM_SPLIT_SEED,
    shuffle=True,
    stratify=random_source["delivery_class_3"],
)
relative_test_fraction = TEST_FRACTION / (VALIDATION_FRACTION + TEST_FRACTION)
validation_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_fraction,
    random_state=RANDOM_SPLIT_SEED,
    shuffle=True,
    stratify=temp_df["delivery_class_3"],
)

train_df = train_df.sample(frac=1, random_state=RANDOM_SPLIT_SEED).reset_index(drop=True)
validation_df = validation_df.sample(frac=1, random_state=RANDOM_SPLIT_SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=RANDOM_SPLIT_SEED).reset_index(drop=True)

train_df["oracle_split"] = "random_train"
validation_df["oracle_split"] = "random_validation"
test_df["oracle_split"] = "random_test"
train_df["cv_fold"] = -1
validation_df["cv_fold"] = -1
test_df["cv_fold"] = -1

train_df.to_csv(RANDOM_TRAIN_PATH, index=False)
validation_df.to_csv(RANDOM_VALIDATION_PATH, index=False)
test_df.to_csv(RANDOM_TEST_PATH, index=False)

split_summary = pd.concat([train_df, validation_df, test_df], ignore_index=True)
split_summary_table = (
    split_summary.groupby(["oracle_split", "delivery_class_3", "delivery_class_label"])
    .agg(rows=("study_id", "size"), studies=("study_id", "nunique"), formulations=("lnp_id", "nunique"))
    .reset_index()
    .sort_values(["oracle_split", "delivery_class_3"])
)
split_summary_table.to_csv(RANDOM_SPLIT_SUMMARY_PATH, index=False)

print("Random split source rows:", f"{len(random_source):,}")
print("Random split source studies:", random_source["study_id"].nunique())
print("Train:", RANDOM_TRAIN_PATH, f"rows={len(train_df):,}")
print("Validation:", RANDOM_VALIDATION_PATH, f"rows={len(validation_df):,}")
print("Test:", RANDOM_TEST_PATH, f"rows={len(test_df):,}")
print("Output folder:", RANDOM_MULTIMODAL_TABULAR_OUT)
print("Using trainer:", TRAINER_PATH)
print("\nRandom split class/study/formulation summary:")
display(split_summary_table)

cmd = [
    sys.executable,
    "-u",
    "-m",
    "lnphub_oracles.train_tabular_multimodal_random_split_classification",
    "--train-subset", str(RANDOM_TRAIN_PATH),
    "--validation-subset", str(RANDOM_VALIDATION_PATH),
    "--test-subset", str(RANDOM_TEST_PATH),
    "--output-dir", str(RANDOM_MULTIMODAL_TABULAR_OUT),
    "--target-col", "delivery_class_3",
    "--multimodal-cols", *MULTIMODAL_FEATURE_COLS,
    "--methods", *TABULAR_MULTIMODAL_METHODS,
]

print("\nRunning:", " ".join(cmd))
print("\n--- live random-split multimodal tabular training output ---\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + env.get("PYTHONPATH", "")
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Random-split multimodal tabular classifier training failed with exit code {return_code}")

manifest_path = RANDOM_MULTIMODAL_TABULAR_OUT / "tabular_multimodal_random_split_manifest.csv"
metrics_path = RANDOM_MULTIMODAL_TABULAR_OUT / "tabular_multimodal_random_split_all_metrics.csv"

print("\nSaved manifest:", manifest_path)
print("Saved all metrics:", metrics_path)

manifest = pd.read_csv(manifest_path)
display(manifest)

print("\nRandom-split test summary:")
display(
    manifest[[
        "method",
        "validation_balanced_accuracy",
        "validation_macro_f1",
        "validation_roc_auc_ovr_macro",
        "validation_average_precision_macro",
        "test_balanced_accuracy",
        "test_macro_f1",
        "test_roc_auc_ovr_macro",
        "test_average_precision_macro",
    ]].sort_values("test_balanced_accuracy", ascending=False)
)


## 9. Random Stratified 5-Fold CV Molecular Fingerprint / Descriptor + Formulation / Physicochemical Classifiers


In [ ]:
# Cell 12 - Random stratified 5-fold CV multimodal tabular classifiers
# This uses the balanced 3-class multimodal dataset, shuffles it,
# holds out a random stratified test set, then performs stratified random 5-fold CV on trainval.
import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display

RANDOM_5FOLD_SEED = 7
N_RANDOM_CV_FOLDS = 5
RANDOM_TEST_FRACTION = 0.15

RANDOM_5FOLD_SPLIT_DIR = OUT_DIR / "classification_random_5fold_splits"
RANDOM_5FOLD_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_5FOLD_TRAINVAL_PATH = RANDOM_5FOLD_SPLIT_DIR / "lnphub_expression_multimodal_3class_balanced_random_trainval_5fold.csv"
RANDOM_5FOLD_TEST_PATH = RANDOM_5FOLD_SPLIT_DIR / "lnphub_expression_multimodal_3class_balanced_random_test.csv"
RANDOM_5FOLD_SUMMARY_PATH = RANDOM_5FOLD_SPLIT_DIR / "random_5fold_split_summary.csv"

RANDOM_5FOLD_TABULAR_OUT = (
    PROJECT_DIR
    / "results"
    / "multimodal_expression_3class_classification_balanced"
    / "molecular_fingerprint_descriptor_random_5fold_cv"
)
RANDOM_5FOLD_TABULAR_OUT.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_DIR / "src"
TRAINER_PATH = SRC_DIR / "lnphub_oracles" / "train_tabular_multimodal_classification.py"
if not TRAINER_PATH.exists():
    print("Multimodal trainer not found; pulling latest dev branch...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", "dev"], check=False)

if not TRAINER_PATH.exists():
    raise FileNotFoundError(
        f"Missing {TRAINER_PATH}. Pull the latest dev branch or rerun the clone/setup cell before training."
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


def ensure_import(import_name, pip_names):
    try:
        __import__(import_name)
        return
    except ModuleNotFoundError:
        pass

    if isinstance(pip_names, str):
        pip_names = [pip_names]

    last_error = None
    for pip_name in pip_names:
        print(f"Installing missing package for {import_name}: {pip_name}")
        res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], text=True, capture_output=True)
        if res.returncode != 0:
            last_error = (res.stdout + res.stderr)[-4000:]
            print(last_error)
            continue
        try:
            __import__(import_name)
            return
        except ModuleNotFoundError as exc:
            last_error = repr(exc)

    raise ModuleNotFoundError(f"Could not import {import_name} after installing {pip_names}. Last error: {last_error}")


ensure_import("rdkit", ["rdkit", "rdkit-pypi"])
ensure_import("pyarrow", "pyarrow")
ensure_import("sklearn", "scikit-learn")
ensure_import("xgboost", "xgboost")

from sklearn.model_selection import StratifiedKFold, train_test_split

MULTIMODAL_FEATURE_COLS = [
    "il_mol_ratio",
    "helper_lipid_mol_ratio",
    "cholesterol_mol_ratio",
    "peg_lipid_mol_ratio",
    "particle_size_nm",
    "pdi",
]

TABULAR_MULTIMODAL_METHODS = [
    "ecfp4_multimodal_mlp",
    "ecfp4_multimodal_xgboost",
    "ecfp4_multimodal_random_forest",
    "rdkit_descriptors_multimodal_mlp",
    "rdkit_descriptors_multimodal_xgboost",
    "rdkit_descriptors_multimodal_random_forest",
]

if "balanced_subset_3class" in globals():
    random_cv_source = balanced_subset_3class.copy()
elif "balanced_subset_path" in globals() and Path(balanced_subset_path).exists():
    random_cv_source = pd.read_csv(balanced_subset_path, low_memory=False)
else:
    candidate_path = OUT_DIR / "lnphub_expression_multimodal_physchem_formulation_3class_balanced_by_study.csv"
    if not candidate_path.exists():
        raise FileNotFoundError("Run the balancing cell first, or provide the saved balanced subset file.")
    random_cv_source = pd.read_csv(candidate_path, low_memory=False)

random_cv_source = random_cv_source.copy()
random_cv_source["delivery_class_3"] = pd.to_numeric(random_cv_source["delivery_class_3"], errors="coerce")
random_cv_source = random_cv_source[random_cv_source["delivery_class_3"].isin([0, 1, 2])].copy()
random_cv_source["delivery_class_3"] = random_cv_source["delivery_class_3"].astype(int)
random_cv_source = random_cv_source.sample(frac=1, random_state=RANDOM_5FOLD_SEED).reset_index(drop=True)

missing_cols = [c for c in MULTIMODAL_FEATURE_COLS if c not in random_cv_source.columns]
if missing_cols:
    raise ValueError(f"Missing multimodal columns in balanced source data: {missing_cols}")

class_counts = random_cv_source["delivery_class_3"].value_counts().sort_index()
if class_counts.min() < N_RANDOM_CV_FOLDS:
    raise ValueError(f"Not enough rows per class for {N_RANDOM_CV_FOLDS}-fold CV: {class_counts.to_dict()}")

trainval_df, test_df = train_test_split(
    random_cv_source,
    test_size=RANDOM_TEST_FRACTION,
    random_state=RANDOM_5FOLD_SEED,
    shuffle=True,
    stratify=random_cv_source["delivery_class_3"],
)
trainval_df = trainval_df.sample(frac=1, random_state=RANDOM_5FOLD_SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=RANDOM_5FOLD_SEED).reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_RANDOM_CV_FOLDS, shuffle=True, random_state=RANDOM_5FOLD_SEED)
trainval_df["cv_fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(trainval_df, trainval_df["delivery_class_3"])):
    trainval_df.loc[val_idx, "cv_fold"] = fold

trainval_df["oracle_split"] = "random_5fold_trainval"
test_df["oracle_split"] = "random_5fold_test"
test_df["cv_fold"] = -1

trainval_df.to_csv(RANDOM_5FOLD_TRAINVAL_PATH, index=False)
test_df.to_csv(RANDOM_5FOLD_TEST_PATH, index=False)

split_summary = pd.concat([trainval_df, test_df], ignore_index=True)
split_summary_table = (
    split_summary.groupby(["oracle_split", "cv_fold", "delivery_class_3", "delivery_class_label"])
    .agg(rows=("study_id", "size"), studies=("study_id", "nunique"), formulations=("lnp_id", "nunique"))
    .reset_index()
    .sort_values(["oracle_split", "cv_fold", "delivery_class_3"])
)
split_summary_table.to_csv(RANDOM_5FOLD_SUMMARY_PATH, index=False)

print("Random 5-fold source rows:", f"{len(random_cv_source):,}")
print("Random 5-fold source studies:", random_cv_source["study_id"].nunique())
print("Trainval:", RANDOM_5FOLD_TRAINVAL_PATH, f"rows={len(trainval_df):,}")
print("Test:", RANDOM_5FOLD_TEST_PATH, f"rows={len(test_df):,}")
print("Output folder:", RANDOM_5FOLD_TABULAR_OUT)
print("Using trainer:", TRAINER_PATH)
print("\nRandom 5-fold split summary:")
display(split_summary_table)

print("\nFold-level class counts:")
display(
    trainval_df.groupby(["cv_fold", "delivery_class_3", "delivery_class_label"])
    .size()
    .reset_index(name="rows")
    .pivot_table(index="cv_fold", columns="delivery_class_label", values="rows", fill_value=0)
    .reset_index()
)

cmd = [
    sys.executable,
    "-u",
    "-m",
    "lnphub_oracles.train_tabular_multimodal_classification",
    "--trainval-subset", str(RANDOM_5FOLD_TRAINVAL_PATH),
    "--test-subset", str(RANDOM_5FOLD_TEST_PATH),
    "--output-dir", str(RANDOM_5FOLD_TABULAR_OUT),
    "--target-col", "delivery_class_3",
    "--cv-description", "stratified random 5-fold CV on balanced trainval data",
    "--multimodal-cols", *MULTIMODAL_FEATURE_COLS,
    "--methods", *TABULAR_MULTIMODAL_METHODS,
]

print("\nRunning:", " ".join(cmd))
print("\n--- live random 5-fold multimodal tabular training output ---\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + env.get("PYTHONPATH", "")
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Random 5-fold multimodal tabular classifier training failed with exit code {return_code}")

manifest_path = RANDOM_5FOLD_TABULAR_OUT / "tabular_multimodal_classification_manifest.csv"
metrics_path = RANDOM_5FOLD_TABULAR_OUT / "tabular_multimodal_all_metrics.csv"

print("\nSaved manifest:", manifest_path)
print("Saved all metrics:", metrics_path)

manifest = pd.read_csv(manifest_path)
display(manifest)

print("\nRandom 5-fold CV + test summary:")
display(
    manifest[[
        "method",
        "cv_balanced_accuracy",
        "cv_macro_f1",
        "cv_roc_auc_ovr_macro",
        "cv_average_precision_macro",
        "test_balanced_accuracy",
        "test_macro_f1",
        "test_roc_auc_ovr_macro",
        "test_average_precision_macro",
    ]].sort_values("test_balanced_accuracy", ascending=False)
)
